# BEVFusion: Multimodal BEV Feature Extractor & Trajectory Planner

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hombosh-dev/SkyNet/blob/feat/bevfusion-encoder/BEVFusion_Encoder.ipynb)

## Overview
This notebook implements an end-to-end multimodal Bird's-Eye-View (BEV) perception and trajectory planning architecture in pure PyTorch:
- **Inputs:** 6 synchronized surround-view cameras (`[B, 6, 3, 128, 352]`) and 3D LiDAR point clouds (`[N, 4]`).
- **Camera BEV Branch:** Pure PyTorch ResNet-18 backbone + learned depth distribution projection into vehicle ego frame (Lift-Splat-Pool).
- **LiDAR BEV Branch:** PointPillars feature network + 2D CNN encoder in ego coordinates.
- **Fusion:** Cross-modal channel concatenation + residual convolution blocks resulting in a unified BEV representation (`[B, 256, 200, 200]`).
- **Downstream Application:** End-to-end `EgoPlanner` predicting 3.0-second future vehicle trajectory waypoints directly from the fused spatial representation.
- **Midterm Benchmarking:** Formal evaluation against standard baselines (**Zero-Motion** and **Constant Velocity**) using official planning metrics (L2 displacement errors at 1s, 2s, 3s horizons).

### Acceptance Criteria
1. Runs out-of-the-box on any standard Colab instance (GPU/CPU) from complete zero without manual dataset setup.
2. Free of external C/SDK dependencies (`nuscenes-devkit`, `pyquaternion`, or custom CUDA kernels) to guarantee reproducibility.
3. Produces a deterministic, fixed-size tensor (`[1, 256, 200, 200]`) with no `NaN` or `Inf` values.


In [ ]:
import os
import sys
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")


In [ ]:
import os
import shutil

DATASET_ROOT = '/content/data/sets/nuscenes'
VERSION = 'v1.0-mini'
META_DIR = os.path.join(DATASET_ROOT, VERSION)

if os.path.exists(os.path.join(META_DIR, 'sample.json')):
    print(f"Dataset already available at: {META_DIR}")
else:
    os.makedirs(DATASET_ROOT, exist_ok=True)

    archive_candidates = [
        '/content/v1.0-mini.tgz',
        '/content/v1.0-mini.tar',
        '/content/drive/MyDrive/v1.0-mini.tgz',
        '/content/drive/MyDrive/v1.0-mini.tar',
        '/content/drive/MyDrive/nuscenes/v1.0-mini.tgz',
        '/content/drive/MyDrive/nuscenes/v1.0-mini.tar',
        './v1.0-mini.tar',
        './v1.0-mini.tgz',
    ]

    if not any(os.path.exists(p) for p in archive_candidates):
        try:
            from google.colab import drive
            if not os.path.exists('/content/drive'):
                print("Checking Google Drive for dataset archive...")
                drive.mount('/content/drive', force_remount=False)
        except Exception:
            pass

    archive_path = next((p for p in archive_candidates if os.path.exists(p)), None)

    if not archive_path:
        archive_path = '/content/v1.0-mini.tgz'
        cdn_url = 'https://www.nuscenes.org/data/v1.0-mini.tgz'
        print(f"Downloading nuScenes {VERSION} directly from CDN (~4 GB)...")
        os.system(f"curl -C - -L -o {archive_path} {cdn_url}")

    print(f"Extracting {archive_path} to {DATASET_ROOT}...")
    os.system(f"tar -xzf {archive_path} -C {DATASET_ROOT} 2>/dev/null || tar -xf {archive_path} -C {DATASET_ROOT}")

    nested_path = os.path.join(DATASET_ROOT, 'mini', VERSION)
    if os.path.exists(nested_path) and not os.path.exists(META_DIR):
        shutil.move(nested_path, DATASET_ROOT)

    assert os.path.exists(os.path.join(META_DIR, 'sample.json')), f"Extraction check failed at: {META_DIR}"
    print(f"Dataset successfully prepared at: {META_DIR}")


In [ ]:
import json
import numpy as np
import torch
from PIL import Image

CAMERAS = [
    'CAM_FRONT_LEFT',
    'CAM_FRONT',
    'CAM_FRONT_RIGHT',
    'CAM_BACK_LEFT',
    'CAM_BACK',
    'CAM_BACK_RIGHT',
]

def quaternion_to_matrix(q):
    """Convert quaternion [w, x, y, z] to a 3x3 rotation matrix."""
    w, x, y, z = q
    return np.array([
        [1 - 2*y*y - 2*z*z, 2*x*y - 2*z*w, 2*x*z + 2*y*w],
        [2*x*y + 2*z*w, 1 - 2*x*x - 2*z*z, 2*y*z - 2*x*w],
        [2*x*z - 2*y*w, 2*y*z + 2*x*w, 1 - 2*x*x - 2*y*y]
    ], dtype=np.float32)

class NuScenesMiniLoader:
    """
    Lightweight, dependency-free loader for nuScenes mini.
    Directly reads JSON metadata and binary sensor files.
    """
    def __init__(self, dataroot=DATASET_ROOT, version=VERSION):
        self.dataroot = dataroot
        self.meta_dir = os.path.join(dataroot, version)
        if not os.path.exists(self.meta_dir):
            sub = os.path.join(dataroot, 'mini', version)
            if os.path.exists(sub):
                self.meta_dir = sub

        with open(os.path.join(self.meta_dir, 'sample.json'), 'r') as f:
            self.samples = json.load(f)
        with open(os.path.join(self.meta_dir, 'sample_data.json'), 'r') as f:
            self.sample_data = json.load(f)
        with open(os.path.join(self.meta_dir, 'calibrated_sensor.json'), 'r') as f:
            self.calibrations = {c['token']: c for c in json.load(f)}
        with open(os.path.join(self.meta_dir, 'ego_pose.json'), 'r') as f:
            self.ego_poses = {e['token']: e for e in json.load(f)}
        with open(os.path.join(self.meta_dir, 'sample_annotation.json'), 'r') as f:
            self.annotations = json.load(f)
        with open(os.path.join(self.meta_dir, 'scene.json'), 'r') as f:
            self.scenes = {s['name']: s for s in json.load(f)}

        self.sample_by_token = {s['token']: s for s in self.samples}
        self.index_by_token = {s['token']: i for i, s in enumerate(self.samples)}
        # Keyframes only: sample_data.json also holds sweeps (LiDAR 20 Hz, cameras 12 Hz)
        # that carry a keyframe's sample_token. Without this filter a sweep could replace
        # the keyframe image / point cloud / ego pose of a sample.
        self.data_by_sample = {}
        for record in self.sample_data:
            if record['is_key_frame']:
                self.data_by_sample.setdefault(record['sample_token'], []).append(record)

    def scene_sample_indices(self, scene_name):
        """Indices of a scene's keyframes in chronological order (follows the 'next' links)."""
        indices, token = [], self.scenes[scene_name]['first_sample_token']
        while token:
            indices.append(self.index_by_token[token])
            token = self.sample_by_token[token]['next']
        return indices

    def ego_pose(self, sample_idx):
        """Ego -> global rotation and translation at the sample's LiDAR keyframe."""
        rec = [r for r in self.data_by_sample[self.samples[sample_idx]['token']] if 'LIDAR_TOP' in r['filename']][0]
        pose = self.ego_poses[rec['ego_pose_token']]
        return quaternion_to_matrix(pose['rotation']), np.array(pose['translation'])

    def ego_velocity(self, sample_idx):
        """Ego velocity (vx, vy) in the current ego frame from the PREVIOUS keyframe to now.
        Uses only the past; None for the first keyframe of a scene."""
        prev_token = self.samples[sample_idx]['prev']
        if not prev_token:
            return None
        prev_idx = self.index_by_token[prev_token]
        r_curr, t_curr = self.ego_pose(sample_idx)
        _, t_prev = self.ego_pose(prev_idx)
        dt = (self.samples[sample_idx]['timestamp'] - self.samples[prev_idx]['timestamp']) * 1e-6
        return (r_curr.T @ (t_curr - t_prev))[:2] / dt

    def load_sample(self, sample_idx=0, img_size=(128, 352)):
        sample = self.samples[sample_idx]
        records = self.data_by_sample[sample['token']]
        sensors = {r['filename'].split('/')[1]: r for r in records}
        root_dir = os.path.dirname(self.meta_dir)

        # 1. Load and normalize 6 surround-view camera views
        imgs, rots, trans, intrins = [], [], [], []
        for cam_name in CAMERAS:
            record = sensors[cam_name]
            cam_path = os.path.join(self.dataroot, record['filename'])
            if not os.path.exists(cam_path):
                cam_path = os.path.join(root_dir, record['filename'])

            img = Image.open(cam_path).convert('RGB')
            orig_w, orig_h = img.size
            target_h, target_w = img_size
            img_resized = img.resize((target_w, target_h), Image.BILINEAR)

            img_np = np.array(img_resized).astype(np.float32) / 255.0
            mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
            std = np.array([0.229, 0.224, 0.225], dtype=np.float32)
            img_np = (img_np - mean) / std
            imgs.append(torch.tensor(img_np, dtype=torch.float32).permute(2, 0, 1))

            cs = self.calibrations[record['calibrated_sensor_token']]
            rots.append(torch.tensor(quaternion_to_matrix(cs['rotation']), dtype=torch.float32))
            trans.append(torch.tensor(cs['translation'], dtype=torch.float32))

            intrin = np.array(cs['camera_intrinsic'], dtype=np.float32)
            intrin[0, :] *= (target_w / orig_w)
            intrin[1, :] *= (target_h / orig_h)
            intrins.append(torch.tensor(intrin, dtype=torch.float32))

        # 2. Load LiDAR point cloud and transform into vehicle ego frame
        lidar_record = sensors['LIDAR_TOP']
        lidar_path = os.path.join(self.dataroot, lidar_record['filename'])
        if not os.path.exists(lidar_path):
            lidar_path = os.path.join(root_dir, lidar_record['filename'])

        raw_pts = np.fromfile(lidar_path, dtype=np.float32).reshape(-1, 5)
        xyz = raw_pts[:, :3]
        intensity = raw_pts[:, 3:4]

        cs_lidar = self.calibrations[lidar_record['calibrated_sensor_token']]
        rot_lidar = quaternion_to_matrix(cs_lidar['rotation'])
        trans_lidar = np.array(cs_lidar['translation'], dtype=np.float32)
        xyz_ego = (rot_lidar @ xyz.T).T + trans_lidar
        lidar_tensor = torch.tensor(np.concatenate([xyz_ego, intensity], axis=1), dtype=torch.float32)

        return {
            'imgs': torch.stack(imgs).unsqueeze(0),
            'rots': torch.stack(rots).unsqueeze(0),
            'trans': torch.stack(trans).unsqueeze(0),
            'intrins': torch.stack(intrins).unsqueeze(0),
            'lidar_points': lidar_tensor,
            'sample_token': sample['token'],
            'sensors': sensors,
        }

    def get_ground_truth_trajectory(self, sample_idx=0, num_steps=6):
        """Extract future trajectory waypoints in the current vehicle ego frame."""
        curr_sample = self.samples[sample_idx]
        lidar_rec = [r for r in self.data_by_sample[curr_sample['token']] if 'LIDAR_TOP' in r['filename']][0]
        curr_ego = self.ego_poses[lidar_rec['ego_pose_token']]
        r_curr = quaternion_to_matrix(curr_ego['rotation'])
        t_curr = np.array(curr_ego['translation'])

        waypoints = []
        curr = curr_sample
        for _ in range(num_steps):
            next_token = curr.get('next', '')
            if not next_token or next_token not in self.sample_by_token:
                break
            curr = self.sample_by_token[next_token]
            next_rec = [r for r in self.data_by_sample[curr['token']] if 'LIDAR_TOP' in r['filename']][0]
            next_ego = self.ego_poses[next_rec['ego_pose_token']]
            t_next = np.array(next_ego['translation'])
            ego_pt = r_curr.T @ (t_next - t_curr)
            waypoints.append(ego_pt[:2])

        return np.array(waypoints) if len(waypoints) > 0 else None

loader = NuScenesMiniLoader()
print(f"Loaded nuScenes dataset: {len(loader.samples)} samples across mini-split.")


In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class BasicBlock(nn.Module):
    def __init__(self, in_planes, planes, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_planes, planes, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != planes:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, planes, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(planes)
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)), inplace=True)
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        return F.relu(out, inplace=True)

class CameraBackbone(nn.Module):
    """ResNet-18 feature extractor backbone (16x downsampling, 256 output channels)."""
    def __init__(self, out_channels=256):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        self.layer1 = nn.Sequential(BasicBlock(64, 64), BasicBlock(64, 64))
        self.layer2 = nn.Sequential(BasicBlock(64, 128, stride=2), BasicBlock(128, 128))
        self.layer3 = nn.Sequential(BasicBlock(128, out_channels, stride=2), BasicBlock(out_channels, out_channels))

    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)), inplace=True)
        x = self.maxpool(x)
        x = self.layer1(x)
        x = self.layer2(x)
        return self.layer3(x)

class QuickVoxelPool(nn.Module):
    """BEV voxel pooling using PyTorch scatter_add_."""
    def __init__(self, nx, ny):
        super().__init__()
        self.nx = nx
        self.ny = ny

    def forward(self, geom_feats, x):
        B, N, D, H, W, C = x.shape
        geom_feats = geom_feats.reshape(B, -1, 3)
        x = x.reshape(B, -1, C)

        outputs = []
        for b in range(B):
            gf = geom_feats[b]
            feat = x[b]
            valid_mask = (
                (gf[:, 0] >= 0) & (gf[:, 0] < self.nx) &
                (gf[:, 1] >= 0) & (gf[:, 1] < self.ny)
            )
            valid_gf = gf[valid_mask]
            valid_feat = feat[valid_mask]

            if len(valid_gf) == 0:
                outputs.append(torch.zeros(C, self.ny, self.nx, device=x.device))
                continue

            flat_idx = (valid_gf[:, 1] * self.nx + valid_gf[:, 0]).long()
            bev_flat = torch.zeros(C, self.ny * self.nx, device=x.device)
            bev_flat.scatter_add_(1, flat_idx.unsqueeze(0).expand(C, -1), valid_feat.T)
            outputs.append(bev_flat.view(C, self.ny, self.nx))

        return torch.stack(outputs, dim=0)

class CameraBEVEncoder(nn.Module):
    """Lifts multi-camera views into vehicle ego BEV coordinates."""
    def __init__(self, xbound=[-50.0, 50.0, 0.5], ybound=[-50.0, 50.0, 0.5],
                 zbound=[-10.0, 10.0, 20.0], dbound=[4.0, 45.0, 1.0], out_channels=64):
        super().__init__()
        self.xbound = xbound
        self.ybound = ybound
        self.zbound = zbound
        self.dbound = dbound
        self.out_channels = out_channels

        self.nx = int((xbound[1] - xbound[0]) / xbound[2])
        self.ny = int((ybound[1] - ybound[0]) / ybound[2])
        self.D = int((dbound[1] - dbound[0]) / dbound[2])

        self.backbone = CameraBackbone(out_channels=256)
        self.depth_conv = nn.Conv2d(256, self.D + out_channels, kernel_size=1)
        self.voxel_pool = QuickVoxelPool(self.nx, self.ny)

    def get_frustum(self, H, W, device):
        fH, fW = H // 16, W // 16
        ds = torch.arange(self.dbound[0], self.dbound[1], self.dbound[2], device=device).float()
        D = ds.shape[0]
        xs = torch.linspace(0, W - 1, fW, device=device).float().view(1, 1, fW).expand(D, fH, fW)
        ys = torch.linspace(0, H - 1, fH, device=device).float().view(1, fH, 1).expand(D, fH, fW)
        ds = ds.view(D, 1, 1).expand(D, fH, fW)
        return torch.stack((xs * ds, ys * ds, ds), -1)

    def forward(self, imgs, rots, trans, intrins):
        B, N, _, H, W = imgs.shape
        device = imgs.device
        imgs = imgs.view(B * N, 3, H, W)
        feats = self.backbone(imgs)

        depth_and_ctx = self.depth_conv(feats)
        depth = F.softmax(depth_and_ctx[:, :self.D], dim=1)
        ctx = depth_and_ctx[:, self.D:]

        x = depth.unsqueeze(1) * ctx.unsqueeze(2)
        x = x.view(B, N, self.out_channels, self.D, H // 16, W // 16).permute(0, 1, 3, 4, 5, 2)

        frustum = self.get_frustum(H, W, device)
        frustum = frustum.unsqueeze(0).unsqueeze(0).repeat(B, N, 1, 1, 1, 1)

        inv_k = torch.inverse(intrins).view(B, N, 1, 1, 1, 3, 3)
        points_cam = torch.matmul(inv_k, frustum.unsqueeze(-1)).squeeze(-1)
        rots = rots.view(B, N, 1, 1, 1, 3, 3)
        trans = trans.view(B, N, 1, 1, 1, 3)
        points_ego = torch.matmul(rots, points_cam.unsqueeze(-1)).squeeze(-1) + trans

        dx = self.xbound[2]
        dy = self.ybound[2]
        geom_x = ((points_ego[..., 0] - self.xbound[0]) / dx).long()
        geom_y = ((points_ego[..., 1] - self.ybound[0]) / dy).long()
        geom_z = torch.zeros_like(geom_x)
        geom = torch.stack((geom_x, geom_y, geom_z), dim=-1)

        return self.voxel_pool(geom, x)


In [ ]:
class LiDARBEVEncoder(nn.Module):
    """PointPillars encoder extracting BEV features from 3D LiDAR point clouds."""
    def __init__(self, xbound=[-50.0, 50.0, 0.5], ybound=[-50.0, 50.0, 0.5],
                 zbound=[-10.0, 10.0, 20.0], in_channels=4, out_channels=64):
        super().__init__()
        self.xbound = xbound
        self.ybound = ybound
        self.zbound = zbound
        self.out_channels = out_channels

        self.nx = int((xbound[1] - xbound[0]) / xbound[2])
        self.ny = int((ybound[1] - ybound[0]) / ybound[2])

        self.pfn = nn.Sequential(
            nn.Linear(in_channels, 32, bias=False),
            nn.BatchNorm1d(32),
            nn.ReLU(inplace=True),
            nn.Linear(32, 64, bias=False),
            nn.BatchNorm1d(64),
            nn.ReLU(inplace=True)
        )

        self.backbone2d = nn.Sequential(
            nn.Conv2d(64, 64, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, points):
        device = points.device
        pts_xyz = points[:, :3]
        pts_feat = points[:, :4]

        mask = (
            (pts_xyz[:, 0] >= self.xbound[0]) & (pts_xyz[:, 0] < self.xbound[1]) &
            (pts_xyz[:, 1] >= self.ybound[0]) & (pts_xyz[:, 1] < self.ybound[1]) &
            (pts_xyz[:, 2] >= self.zbound[0]) & (pts_xyz[:, 2] < self.zbound[1])
        )
        valid_pts = pts_feat[mask]
        if len(valid_pts) == 0:
            return torch.zeros((1, self.out_channels, self.ny, self.nx), device=device)

        col_x = ((valid_pts[:, 0] - self.xbound[0]) / self.xbound[2]).long()
        col_y = ((valid_pts[:, 1] - self.ybound[0]) / self.ybound[2]).long()
        pillar_indices = col_y * self.nx + col_x

        feats = self.pfn(valid_pts)
        pillar_map = torch.zeros((64, self.ny * self.nx), device=device)
        pillar_map.scatter_reduce_(
            dim=1,
            index=pillar_indices.unsqueeze(0).expand(64, -1),
            src=feats.T,
            reduce='amax',
            include_self=False
        )

        pillar_img = pillar_map.view(1, 64, self.ny, self.nx)
        return self.backbone2d(pillar_img)


In [ ]:
class ResBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels, channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(channels)
        )
    def forward(self, x):
        return F.relu(x + self.conv(x), inplace=True)

class BEVFusionEncoder(nn.Module):
    """
    Multimodal BEVFusion architecture:
    Fuses camera and LiDAR BEV feature representations into a unified spatial tensor.
    """
    def __init__(self, xbound=[-50.0, 50.0, 0.5], ybound=[-50.0, 50.0, 0.5],
                 zbound=[-10.0, 10.0, 20.0], out_channels=256):
        super().__init__()
        self.cam_channels = 64
        self.lidar_channels = 64

        self.camera_encoder = CameraBEVEncoder(
            xbound=xbound, ybound=ybound, zbound=zbound, out_channels=self.cam_channels
        )
        self.lidar_encoder = LiDARBEVEncoder(
            xbound=xbound, ybound=ybound, zbound=zbound, out_channels=self.lidar_channels
        )

        fused_in_dim = self.cam_channels + self.lidar_channels
        self.fusion = nn.Sequential(
            nn.Conv2d(fused_in_dim, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            ResBlock(out_channels),
            ResBlock(out_channels)
        )

    def forward(self, imgs, rots, trans, intrins, lidar_points):
        cam_bev = self.camera_encoder(imgs, rots, trans, intrins)
        lid_bev = self.lidar_encoder(lidar_points)
        fused_input = torch.cat([cam_bev, lid_bev], dim=1)
        fused_bev = self.fusion(fused_input)
        return fused_bev, cam_bev, lid_bev


In [ ]:
import time

torch.manual_seed(42)  # the encoder weights are random: fix them so every run gives the same numbers
model = BEVFusionEncoder().to(device)
model.eval()

sample = loader.load_sample(sample_idx=0)
imgs = sample['imgs'].to(device)
rots = sample['rots'].to(device)
trans = sample['trans'].to(device)
intrins = sample['intrins'].to(device)
lidar_points = sample['lidar_points'].to(device)

print("Executing forward pass...")
start_time = time.time()
with torch.no_grad():
    fused_tensor, cam_bev, lid_bev = model(imgs, rots, trans, intrins, lidar_points)
latency_ms = (time.time() - start_time) * 1000

expected_shape = (1, 256, 200, 200)
assert fused_tensor.shape == expected_shape, f"Shape mismatch: expected {expected_shape}, got {fused_tensor.shape}"
assert not torch.isnan(fused_tensor).any(), "Output tensor contains NaN values"
assert not torch.isinf(fused_tensor).any(), "Output tensor contains Inf values"

print("\n" + "=" * 50)
print("ACCEPTANCE CRITERIA RESULTS:")
print("=" * 50)
print(f"Output tensor shape: {tuple(fused_tensor.shape)} (fixed dimension)")
print(f"NaN check:           PASSED (no NaNs)")
print(f"Inf check:           PASSED (no Infs)")
print(f"Inference latency:   {latency_ms:.2f} ms")
print(f"Value range:         [{fused_tensor.min().item():.3f}, {fused_tensor.max().item():.3f}]")
print(f"Mean / Std:          {fused_tensor.mean().item():.3f} / {fused_tensor.std().item():.3f}")
if torch.cuda.is_available():
    print(f"GPU VRAM allocated:  {torch.cuda.memory_allocated() / (1024**2):.2f} MB")
print("=" * 50)


In [ ]:
import matplotlib.pyplot as plt

cam_map = cam_bev[0].mean(dim=0).cpu().numpy()
lid_map = lid_bev[0].mean(dim=0).cpu().numpy()
fused_map = fused_tensor[0].mean(dim=0).cpu().numpy()

fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))

im0 = axes[0].imshow(cam_map, cmap='viridis', origin='lower')
axes[0].set_title("Camera BEV Features (Lift-Splat)", fontsize=12, fontweight='bold')
axes[0].axis('off')
plt.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)

im1 = axes[1].imshow(lid_map, cmap='magma', origin='lower')
axes[1].set_title("LiDAR BEV Features (PointPillars)", fontsize=12, fontweight='bold')
axes[1].axis('off')
plt.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)

im2 = axes[2].imshow(fused_map, cmap='plasma', origin='lower')
axes[2].set_title("Fused Multimodal BEV Representation", fontsize=12, fontweight='bold')
axes[2].axis('off')
plt.colorbar(im2, ax=axes[2], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.savefig('/content/bev_features_visualization.png', dpi=120)
plt.show()


In [ ]:
class EgoPlanner(nn.Module):
    """Downstream trajectory planner forecasting future vehicle waypoints.

    Inputs: the fused BEV tensor and the ego velocity measured from the previous keyframe
    (a single frame says nothing about the current speed). The head predicts a correction
    to constant-velocity extrapolation of that past velocity.
    """
    def __init__(self, in_channels=256, num_waypoints=6):
        super().__init__()
        self.num_waypoints = num_waypoints
        self.pool = nn.AdaptiveAvgPool2d((25, 25))  # 200x200 BEV -> 25x25 (4 m cells)
        self.encoder = nn.Sequential(
            nn.Conv2d(in_channels, 128, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 64, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 32, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((1, 1))
        )
        self.mlp = nn.Sequential(
            nn.Linear(32 + 2, 128),
            nn.ReLU(inplace=True),
            nn.Linear(128, 64),
            nn.ReLU(inplace=True),
            nn.Linear(64, num_waypoints * 2)
        )
        self.register_buffer('time_steps', torch.arange(1, num_waypoints + 1).float() * 0.5)

    def head(self, pooled_bev, ego_vel):
        B = pooled_bev.shape[0]
        feat = self.encoder(pooled_bev).flatten(1)
        residual = self.mlp(torch.cat([feat, ego_vel / 10.0], dim=1)).view(B, self.num_waypoints, 2)
        cv = self.time_steps.view(1, -1, 1) * ego_vel.view(B, 1, 2)
        return cv + residual

    def forward(self, fused_bev, ego_vel):
        return self.head(self.pool(fused_bev), ego_vel)


# Train / evaluate on the official nuScenes mini splits (the earlier version had no training
# and replaced the prediction with ground truth + random noise).
TRAIN_SCENES = ['scene-0061', 'scene-0553', 'scene-0655', 'scene-0757',
                'scene-0796', 'scene-1077', 'scene-1094', 'scene-1100']
VAL_SCENES = ['scene-0103', 'scene-0916']


def planner_windows(scene_names):
    """Keyframes with a previous keyframe (for the velocity) and 6 future keyframes (for GT)."""
    out = []
    for name in scene_names:
        for k, idx in enumerate(loader.scene_sample_indices(name)):
            gt = loader.get_ground_truth_trajectory(sample_idx=idx, num_steps=6)
            if k > 0 and gt is not None and len(gt) == 6:
                out.append(idx)
    return out


@torch.no_grad()
def planner_inputs(sample_idx):
    s = loader.load_sample(sample_idx=sample_idx)
    f_bev, _, _ = model(s['imgs'].to(device), s['rots'].to(device), s['trans'].to(device),
                        s['intrins'].to(device), s['lidar_points'].to(device))
    vel = torch.tensor(loader.ego_velocity(sample_idx), dtype=torch.float32).view(1, 2)
    return planner.pool(f_bev).cpu(), vel


torch.manual_seed(42)
planner = EgoPlanner().to(device)

# The encoder has no pretrained checkpoint and stays frozen; only the planner is trained.
train_idx = planner_windows(TRAIN_SCENES)
print(f"Encoding {len(train_idx)} training keyframes from {len(TRAIN_SCENES)} mini_train scenes...")
train_inputs = [planner_inputs(i) for i in train_idx]
X_bev = torch.cat([x for x, _ in train_inputs])
X_vel = torch.cat([v for _, v in train_inputs])
Y = torch.tensor(np.stack([loader.get_ground_truth_trajectory(sample_idx=i, num_steps=6) for i in train_idx]),
                 dtype=torch.float32)

optimizer = torch.optim.Adam(planner.parameters(), lr=1e-3)
EPOCHS, BATCH = 100, 32
gen = torch.Generator().manual_seed(42)
planner.train()
for epoch in range(EPOCHS):
    perm = torch.randperm(len(Y), generator=gen)
    losses = []
    for b in range(0, len(Y), BATCH):
        sel = perm[b:b + BATCH]
        if len(sel) < 2:
            continue
        pred = planner.head(X_bev[sel].to(device), X_vel[sel].to(device))
        loss = (pred - Y[sel].to(device)).abs().mean()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    if (epoch + 1) % 20 == 0:
        print(f"  epoch {epoch + 1:3d}/{EPOCHS}: train L1 {np.mean(losses):.3f} m")
planner.eval()

# Demo keyframe from a held-out mini_val scene (with a previous keyframe for the velocity).
demo_idx = loader.scene_sample_indices(VAL_SCENES[0])[10]
sample = loader.load_sample(sample_idx=demo_idx)
with torch.no_grad():
    fused_tensor, cam_bev, lid_bev = model(sample['imgs'].to(device), sample['rots'].to(device),
                                           sample['trans'].to(device), sample['intrins'].to(device),
                                           sample['lidar_points'].to(device))
    demo_vel = torch.tensor(loader.ego_velocity(demo_idx), dtype=torch.float32, device=device).view(1, 2)
    pred_waypoints = planner(fused_tensor, demo_vel)[0].cpu().numpy()

gt_waypoints = loader.get_ground_truth_trajectory(sample_idx=demo_idx, num_steps=6)

print(f"Predicted 3.0-second vehicle trajectory waypoints ({VAL_SCENES[0]}, keyframe 10):")
for i, (x_fwd, y_lat) in enumerate(pred_waypoints):
    t = (i + 1) * 0.5
    print(f"  +{t:.1f}s -> Forward: {x_fwd:6.2f} m | Lateral: {y_lat:6.2f} m")


In [ ]:
import matplotlib.patches as patches

cam_record = sample['sensors']['CAM_FRONT']
cam_path = os.path.join(loader.dataroot, cam_record['filename'])
if not os.path.exists(cam_path):
    cam_path = os.path.join(os.path.dirname(loader.meta_dir), cam_record['filename'])
front_img = Image.open(cam_path).convert('RGB')
orig_w, orig_h = front_img.size

cs_cam = loader.calibrations[cam_record['calibrated_sensor_token']]
r_cam = quaternion_to_matrix(cs_cam['rotation'])
t_cam = np.array(cs_cam['translation'])
k_cam = np.array(cs_cam['camera_intrinsic'])

curr_ego_pose = loader.ego_poses[cam_record['ego_pose_token']]
r_ego = quaternion_to_matrix(curr_ego_pose['rotation'])
t_ego = np.array(curr_ego_pose['translation'])

def project_ego_to_image(waypoints):
    pixels = []
    for pt in waypoints:
        p_ego = np.array([pt[0], pt[1], -1.0])
        p_cam = r_cam.T @ (p_ego - t_cam)
        if p_cam[2] > 0:
            u = (k_cam[0, 0] * p_cam[0] + k_cam[0, 2] * p_cam[2]) / p_cam[2]
            v = (k_cam[1, 1] * p_cam[1] + k_cam[1, 2] * p_cam[2]) / p_cam[2]
            if 0 <= u < orig_w and 0 <= v < orig_h:
                pixels.append((u, v))
    return pixels

cam_pixels = project_ego_to_image(pred_waypoints)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Panel 1: Front camera view with AR trajectory overlay & 3D obstacles
axes[0].imshow(front_img)

sample_anns = [a for a in loader.annotations if a['sample_token'] == sample['sample_token']]
for ann in sample_anns:
    pos_world = np.array(ann['translation'])
    pos_ego = r_ego.T @ (pos_world - t_ego)
    pos_cam = r_cam.T @ (pos_ego - t_cam)
    if pos_cam[2] > 3.0:
        w_y, l_x, h_z = ann['size']
        u = (k_cam[0, 0] * pos_cam[0] + k_cam[0, 2] * pos_cam[2]) / pos_cam[2]
        v = (k_cam[1, 1] * pos_cam[1] + k_cam[1, 2] * pos_cam[2]) / pos_cam[2]
        bw = (k_cam[0, 0] * w_y) / pos_cam[2]
        bh = (k_cam[1, 1] * h_z) / pos_cam[2]
        if 0 <= u < orig_w and 0 <= v < orig_h:
            box_col = '#00e5ff' if l_x > 2.0 else '#ffaa00'
            axes[0].add_patch(patches.Rectangle((u - bw/2, v - bh/2), bw, bh, linewidth=2, edgecolor=box_col, facecolor='none'))
            axes[0].text(u - bw/2, max(15, v - bh/2 - 4), f"{pos_cam[2]:.0f}m", color='white', fontsize=8, fontweight='bold',
                         bbox=dict(boxstyle='round,pad=0.2', facecolor=box_col, alpha=0.7, edgecolor='none'))

if len(cam_pixels) > 1:
    pxs, pys = zip(*cam_pixels)
    axes[0].plot(pxs, pys, color='#00ff44', linewidth=4, marker='o', markersize=8, label='Planned Path (EgoPlanner)')

axes[0].set_title("1. Front Camera View with AR Trajectory & 3D Obstacles", fontsize=12, fontweight='bold')
h_0, l_0 = axes[0].get_legend_handles_labels()
if h_0:
    axes[0].legend(loc='lower left', fontsize=10)
axes[0].axis('off')

# Panel 2: Bird's-Eye-View scene map
axes[1].set_facecolor('#11131a')
axes[1].set_xlim(-15, 35)
axes[1].set_ylim(-25, 25)

# Background LiDAR scan for road boundaries & buildings
pts_ego = sample['lidar_points'].cpu().numpy()
m_lidar = (pts_ego[:, 0] >= -15) & (pts_ego[:, 0] <= 35) & (pts_ego[:, 1] >= -25) & (pts_ego[:, 1] <= 25)
axes[1].scatter(pts_ego[m_lidar, 0], pts_ego[m_lidar, 1], s=0.8, color='#343d52', alpha=0.5, label='LiDAR Scan')

# Ego vehicle (properly oriented: length 4.4m forward, width 2.0m lateral)
axes[1].add_patch(patches.Rectangle((-2.2, -1.0), 4.4, 2.0, color='#ff3333', label='Ego Vehicle', zorder=5))
axes[1].arrow(0, 0, 3.2, 0, head_width=1.2, head_length=1.2, fc='white', ec='red', zorder=6)

# Surrounding detected obstacles
drawn_veh = False
for ann in sample_anns:
    pos_world = np.array(ann['translation'])
    pos_ego = r_ego.T @ (pos_world - t_ego)
    if -15 <= pos_ego[0] <= 35 and -25 <= pos_ego[1] <= 25:
        w_y, l_x, _ = ann['size']
        is_veh = l_x > 2.0
        c = '#00e5ff' if is_veh else '#ffaa00'
        lbl = 'Detected Obstacles' if not drawn_veh else None
        axes[1].add_patch(patches.Rectangle((pos_ego[0] - l_x/2, pos_ego[1] - w_y/2), l_x, w_y,
                                            color=c, alpha=0.75, linewidth=1.2, edgecolor='white', zorder=4, label=lbl))
        drawn_veh = True

axes[1].plot(pred_waypoints[:, 0], pred_waypoints[:, 1], color='#00ff44', linewidth=3,
             marker='o', markersize=7, label='Planned Trajectory (AI)', zorder=7)
if gt_waypoints is not None:
    axes[1].plot(gt_waypoints[:, 0], gt_waypoints[:, 1], color='yellow', linestyle='--',
                 linewidth=2.5, marker='x', markersize=6, label='Human Driver (GT)', zorder=7)

axes[1].set_title("2. BEV Scene Map with Obstacles & Trajectories", fontsize=12, fontweight='bold')
axes[1].set_xlabel("Forward (meters)")
axes[1].set_ylabel("Lateral (meters)")
axes[1].grid(True, linestyle='--', alpha=0.3)
axes[1].legend(loc='upper left', fontsize=10)

plt.tight_layout()
plt.savefig('/content/ego_planner_trajectory.png', dpi=120)
plt.show()


In [ ]:
from IPython.display import Image as IPImage, display

GIF_SCENE = VAL_SCENES[0]  # held-out scene, never seen by the planner during training
scene_indices = loader.scene_sample_indices(GIF_SCENE)  # chronological keyframes
max_frames = min(len(scene_indices) - 2, 38)

print(f"Rendering multi-frame driving simulation across {max_frames} frames...")
gif_frames = []

for frame_idx, sample_idx in enumerate(scene_indices[:max_frames]):
    ego_vel = loader.ego_velocity(sample_idx)
    if ego_vel is None:  # first keyframe: no previous frame to measure the speed from
        continue
    sample_data = loader.load_sample(sample_idx=sample_idx)
    imgs_t = sample_data['imgs'].to(device)
    rots_t = sample_data['rots'].to(device)
    trans_t = sample_data['trans'].to(device)
    intrins_t = sample_data['intrins'].to(device)
    pts_t = sample_data['lidar_points'].to(device)

    with torch.no_grad():
        f_bev, _, _ = model(imgs_t, rots_t, trans_t, intrins_t, pts_t)
        vel_t = torch.tensor(ego_vel, dtype=torch.float32, device=device).view(1, 2)
        pred_wp = planner(f_bev, vel_t)[0].cpu().numpy()

    gt_wp = loader.get_ground_truth_trajectory(sample_idx=sample_idx, num_steps=6)

    cam_rec = sample_data['sensors']['CAM_FRONT']
    c_path = os.path.join(loader.dataroot, cam_rec['filename'])
    if not os.path.exists(c_path):
        c_path = os.path.join(os.path.dirname(loader.meta_dir), cam_rec['filename'])
    f_img = Image.open(c_path).convert('RGB')

    cs = loader.calibrations[cam_rec['calibrated_sensor_token']]
    r_c = quaternion_to_matrix(cs['rotation'])
    t_c = np.array(cs['translation'])
    k_c = np.array(cs['camera_intrinsic'])

    curr_ego = loader.ego_poses[cam_rec['ego_pose_token']]
    r_ego = quaternion_to_matrix(curr_ego['rotation'])
    t_ego = np.array(curr_ego['translation'])

    def project_wp(waypoints):
        pxs = []
        for pt in waypoints:
            p_cam = r_c.T @ (np.array([pt[0], pt[1], -1.0]) - t_c)
            if p_cam[2] > 0:
                u = (k_c[0, 0] * p_cam[0] + k_c[0, 2] * p_cam[2]) / p_cam[2]
                v = (k_c[1, 1] * p_cam[1] + k_c[1, 2] * p_cam[2]) / p_cam[2]
                if 0 <= u < f_img.size[0] and 0 <= v < f_img.size[1]:
                    pxs.append((u, v))
        return pxs

    pred_pxs = project_wp(pred_wp)
    gt_pxs = project_wp(gt_wp) if gt_wp is not None else []

    fig, axs = plt.subplots(1, 2, figsize=(16, 7))
    axs[0].imshow(f_img)

    # 1. Project obstacle 3D bounding boxes onto front camera
    sample_anns = [a for a in loader.annotations if a['sample_token'] == sample_data['sample_token']]
    for ann in sample_anns:
        pos_world = np.array(ann['translation'])
        pos_ego = r_ego.T @ (pos_world - t_ego)
        pos_cam = r_c.T @ (pos_ego - t_c)
        if pos_cam[2] > 3.0:
            w_y, l_x, h_z = ann['size']
            u = (k_c[0, 0] * pos_cam[0] + k_c[0, 2] * pos_cam[2]) / pos_cam[2]
            v = (k_c[1, 1] * pos_cam[1] + k_c[1, 2] * pos_cam[2]) / pos_cam[2]
            bw = (k_c[0, 0] * w_y) / pos_cam[2]
            bh = (k_c[1, 1] * h_z) / pos_cam[2]
            if 0 <= u < f_img.size[0] and 0 <= v < f_img.size[1]:
                box_col = '#00e5ff' if l_x > 2.0 else '#ffaa00'
                axs[0].add_patch(patches.Rectangle((u - bw/2, v - bh/2), bw, bh, linewidth=2, edgecolor=box_col, facecolor='none'))
                axs[0].text(u - bw/2, max(15, v - bh/2 - 4), f"{pos_cam[2]:.0f}m", color='white', fontsize=8, fontweight='bold',
                            bbox=dict(boxstyle='round,pad=0.2', facecolor=box_col, alpha=0.7, edgecolor='none'))

    if len(gt_pxs) > 1:
        axs[0].plot(*zip(*gt_pxs), color='#ffee00', linestyle='--', linewidth=3, marker='x', label='Human Driver (GT)')
    if len(pred_pxs) > 1:
        axs[0].plot(*zip(*pred_pxs), color='#00ff44', linewidth=4, marker='o', label='Planned Path (AI)')
    axs[0].set_title(f"Front Camera AR Overlay (Frame {frame_idx + 1}/{max_frames})", fontsize=12, fontweight='bold')
    h_c, l_c = axs[0].get_legend_handles_labels()
    if h_c:
        axs[0].legend(loc='lower left', fontsize=10)
    axs[0].axis('off')

    # 2. Panel 2: BEV Scene Map with obstacles and LiDAR
    axs[1].set_facecolor('#11131a')
    axs[1].set_xlim(-15, 35)
    axs[1].set_ylim(-25, 25)

    # A. Background LiDAR scan points
    pts_np = pts_t.cpu().numpy()
    m_lid = (pts_np[:, 0] >= -15) & (pts_np[:, 0] <= 35) & (pts_np[:, 1] >= -25) & (pts_np[:, 1] <= 25)
    axs[1].scatter(pts_np[m_lid, 0], pts_np[m_lid, 1], s=0.8, color='#343d52', alpha=0.5, label='LiDAR Scan')

    # B. Ego Vehicle (properly oriented: length 4.4m, width 2.0m)
    axs[1].add_patch(patches.Rectangle((-2.2, -1.0), 4.4, 2.0, color='#ff3333', label='Ego Vehicle', zorder=5))
    axs[1].arrow(0, 0, 3.2, 0, head_width=1.2, head_length=1.2, fc='white', ec='red', zorder=6)

    # C. Surrounding detected obstacles (vehicles in cyan, pedestrians/barriers in orange)
    drawn_v = False
    for ann in sample_anns:
        pos_world = np.array(ann['translation'])
        pos_ego = r_ego.T @ (pos_world - t_ego)
        if -15 <= pos_ego[0] <= 35 and -25 <= pos_ego[1] <= 25:
            w_y, l_x, _ = ann['size']
            is_v = l_x > 2.0
            col = '#00e5ff' if is_v else '#ffaa00'
            lbl = 'Detected Obstacles' if not drawn_v else None
            axs[1].add_patch(patches.Rectangle((pos_ego[0] - l_x/2, pos_ego[1] - w_y/2), l_x, w_y,
                                               color=col, alpha=0.75, linewidth=1.2, edgecolor='white', zorder=4, label=lbl))
            drawn_v = True

    axs[1].plot(pred_wp[:, 0], pred_wp[:, 1], color='#00ff44', linewidth=3, marker='o', label='Planned (AI)', zorder=7)
    if gt_wp is not None:
        axs[1].plot(gt_wp[:, 0], gt_wp[:, 1], color='#ffee00', linestyle='--', linewidth=2.5, marker='x', label='Driver (GT)', zorder=7)

    axs[1].set_title("BEV Scene Map with Obstacles & LiDAR", fontsize=12, fontweight='bold')
    axs[1].set_xlabel("Forward (m)")
    axs[1].set_ylabel("Lateral (m)")
    axs[1].grid(True, linestyle='--', alpha=0.3)
    axs[1].legend(loc='upper left', fontsize=10)

    plt.tight_layout()
    tmp_frame = f'/tmp/frame_{frame_idx:03d}.png'
    plt.savefig(tmp_frame, dpi=100)
    plt.close(fig)
    gif_frames.append(Image.open(tmp_frame))

gif_output_path = f'/content/{GIF_SCENE.replace("-", "")}_full_driving_planning.gif'
gif_frames[0].save(
    gif_output_path,
    save_all=True,
    append_images=gif_frames[1:],
    duration=250,
    loop=0
)
print(f"Simulation saved to: {gif_output_path}")
display(IPImage(filename=gif_output_path, width=800))


In [ ]:
# ==============================================================================
# Baseline Evaluation & Planning Metrics Benchmark
# Matches src/skynet/eval/planning_metrics.py standard: L2 error at 1s, 2s, 3s, avg
# ==============================================================================

def compute_planning_metrics(preds, gts):
    """
    Compute L2 displacement error (in meters) at 1s, 2s, 3s, and average.
    NuScenes keyframes are 2 Hz (0.5s interval):
      - 1s horizon: step index 1 (time = +1.0s)
      - 2s horizon: step index 3 (time = +2.0s)
      - 3s horizon: step index 5 (time = +3.0s)
    """
    pred_arr = np.array(preds)
    gt_arr = np.array(gts)
    
    # Euclidean distance per step: shape [N_samples, 6]
    dists = np.linalg.norm(pred_arr - gt_arr, axis=-1)
    mean_per_step = dists.mean(axis=0)
    
    l2_1s = float(mean_per_step[1])
    l2_2s = float(mean_per_step[3])
    l2_3s = float(mean_per_step[5])
    l2_avg = float((l2_1s + l2_2s + l2_3s) / 3.0)
    
    return {
        "L2 @ 1s (m)": l2_1s,
        "L2 @ 2s (m)": l2_2s,
        "L2 @ 3s (m)": l2_3s,
        "L2 Avg (m)": l2_avg,
    }

# Gather trajectories across all valid keyframes of the held-out mini_val scenes
val_idx = planner_windows(VAL_SCENES)
print(f"Evaluating on {len(val_idx)} keyframes of {VAL_SCENES} (held out from planner training)")
all_gts = []
all_zero_motion = []
all_const_vel = []
all_model_preds = []

time_steps = np.array([0.5, 1.0, 1.5, 2.0, 2.5, 3.0])

for idx in val_idx:
    gt_t = loader.get_ground_truth_trajectory(sample_idx=idx, num_steps=6)
    if gt_t is None or len(gt_t) < 6:
        continue
    all_gts.append(gt_t)
    
    # 1. Zero-Motion Baseline (Static vehicle: assumes vehicle remains at 0, 0)
    all_zero_motion.append(np.zeros((6, 2)))
    
    # 2. Constant Velocity Baseline: velocity from the previous keyframe to now (past only;
    #    the earlier version took it from the first future waypoint, i.e. from the ground truth)
    v_past = loader.ego_velocity(idx)
    all_const_vel.append(np.outer(time_steps, v_past))
    
    # 3. Model Prediction (BEVFusion + EgoPlanner), trained on mini_train only
    pooled, vel = planner_inputs(idx)
    with torch.no_grad():
        pred_wp = planner.head(pooled.to(device), vel.to(device))[0].cpu().numpy()
    all_model_preds.append(pred_wp)

metrics_zero = compute_planning_metrics(all_zero_motion, all_gts)
metrics_vel = compute_planning_metrics(all_const_vel, all_gts)
metrics_model = compute_planning_metrics(all_model_preds, all_gts)

benchmark_rows = [
    {"Method / Model": "Zero-Motion (Static Baseline)", **metrics_zero, "Description": "Ego remains stationary (x=0, y=0)"},
    {"Method / Model": "Constant Velocity (Linear)", **metrics_vel, "Description": "Linear extrapolation of the last observed velocity"},
    {"Method / Model": "BEVFusion + EgoPlanner (Ours)", **metrics_model, "Description": "Multimodal BEV + ego velocity, planner trained on mini_train"},
]

print("\n" + "=" * 80)
print("             MIDTERM DEFENSE BENCHMARK: PLANNING METRICS")
print("=" * 80)
print(f"{'Method / Model':<32} | {'L2 @ 1s':<9} | {'L2 @ 2s':<9} | {'L2 @ 3s':<9} | {'L2 Avg':<9}")
print("-" * 80)
for row in benchmark_rows:
    print(f"{row['Method / Model']:<32} | {row['L2 @ 1s (m)']:<9.2f} | {row['L2 @ 2s (m)']:<9.2f} | {row['L2 @ 3s (m)']:<9.2f} | {row['L2 Avg (m)']:<9.2f}")
print("=" * 80)
print("* L2 displacement error in meters (lower is better).")
print("* Official evaluation metrics aligned with UniAD / VAD benchmarks and SkyNet project schema.")


In [ ]:
# Comparative Visualization: Baselines vs BEVFusion EgoPlanner
fig, ax = plt.subplots(figsize=(9, 8))
ax.set_facecolor('#11131a')

# Ego vehicle footprint
ax.add_patch(patches.Rectangle((-1.0, -2.2), 2.0, 4.4, color='#ff3333', label='Ego Vehicle'))
ax.arrow(0, 0, 3, 0, head_width=1.5, head_length=1.5, fc='red', ec='white')

# Representative sample comparison
sample_idx = 0
ax.plot(all_zero_motion[sample_idx][:, 0], all_zero_motion[sample_idx][:, 1],
        color='#888888', linestyle=':', marker='o', markersize=6, label='Zero-Motion Baseline')
ax.plot(all_const_vel[sample_idx][:, 0], all_const_vel[sample_idx][:, 1],
        color='#3399ff', linestyle='-.', linewidth=2, marker='s', markersize=6, label='Constant Velocity Baseline')
ax.plot(all_model_preds[sample_idx][:, 0], all_model_preds[sample_idx][:, 1],
        color='#00ff44', linewidth=3, marker='o', markersize=7, label='BEVFusion + EgoPlanner (Ours)')
ax.plot(all_gts[sample_idx][:, 0], all_gts[sample_idx][:, 1],
        color='#ffee00', linestyle='--', linewidth=2.5, marker='x', markersize=7, label='Human Driver (Ground Truth)')

ax.set_xlim(-10, 35)
ax.set_ylim(-20, 20)
ax.set_title("Trajectory Comparison: Baselines vs BEVFusion EgoPlanner", fontsize=13, fontweight='bold')
ax.set_xlabel("Forward Distance (meters)", fontsize=11)
ax.set_ylabel("Lateral Offset (meters)", fontsize=11)
ax.grid(True, linestyle='--', alpha=0.3)
ax.legend(loc='upper left', fontsize=10)

plt.tight_layout()
plt.savefig('/content/baselines_comparison.png', dpi=120)
plt.show()
print("Saved baseline comparison plot to: /content/baselines_comparison.png")
